# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*I will use a transparent refresh-priority rule based on recent changes in impressions, clicks, and sessions. A content item receives a higher score when these signals decline from the previous 30-day window. The score is used only to rank items for human review. Reason codes explain which signals contributed to the priority.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Recent change compared with the previous 30-day window
df["impressions_change"] = (
    df["impressions_last_30d"] - df["impressions_prev_30d"]
) / df["impressions_prev_30d"].replace(0, np.nan)

df["clicks_change"] = (
    df["clicks_last_30d"] - df["clicks_prev_30d"]
) / df["clicks_prev_30d"].replace(0, np.nan)

df["sessions_change"] = (
    df["sessions_last_30d"] - df["sessions_prev_30d"]
) / df["sessions_prev_30d"].replace(0, np.nan)

# Convert declines into positive priority signals
df["impression_decline"] = (-df["impressions_change"]).clip(lower=0)
df["click_decline"] = (-df["clicks_change"]).clip(lower=0)
df["session_decline"] = (-df["sessions_change"]).clip(lower=0)

# Cap extreme values so one outlier does not dominate the score
df["impression_decline"] = df["impression_decline"].clip(upper=1)
df["click_decline"] = df["click_decline"].clip(upper=1)
df["session_decline"] = df["session_decline"].clip(upper=1)

# Transparent baseline score
df["baseline_score"] = (
    0.4 * df["impression_decline"]
    + 0.4 * df["click_decline"]
    + 0.2 * df["session_decline"]
)

print("Baseline rule created.")
print("Rows scored:", len(df))
print("Score range:", round(df["baseline_score"].min(), 3),
      "to", round(df["baseline_score"].max(), 3))

Baseline rule created.
Rows scored: 30000
Score range: 0.0 to 1.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
I will rank all content items from highest to lowest baseline refresh score. Each item will receive an action and reason codes so that the output can be used as a review queue rather than an automatic decision.

In [7]:
# Build the ranked queue
baseline = df.copy()

# Replace invalid/infinite changes with 0
change_cols = [
    "impressions_change",
    "clicks_change",
    "sessions_change"
]

for col in change_cols:
    baseline[col] = baseline[col].replace([np.inf, -np.inf], np.nan).fillna(0)

# Recalculate decline signals
baseline["impression_decline"] = (-baseline["impressions_change"]).clip(lower=0, upper=1)
baseline["click_decline"] = (-baseline["clicks_change"]).clip(lower=0, upper=1)
baseline["session_decline"] = (-baseline["sessions_change"]).clip(lower=0, upper=1)

# Transparent baseline score
baseline["baseline_score"] = (
    0.4 * baseline["impression_decline"]
    + 0.4 * baseline["click_decline"]
    + 0.2 * baseline["session_decline"]
)

# Make absolutely sure the score contains no NaN or infinity
baseline["baseline_score"] = (
    baseline["baseline_score"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Rank all content items
baseline["rank"] = (
    baseline["baseline_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Assign action
baseline["action"] = np.select(
    [
        baseline["baseline_score"] >= 0.60,
        baseline["baseline_score"] >= 0.30
    ],
    [
        "Review / Refresh",
        "Review"
    ],
    default="Monitor"
)

# Create reason codes
def get_reason_codes(row):
    reasons = []

    if row["impression_decline"] >= 0.20:
        reasons.append("IMPRESSION_DECLINE")

    if row["click_decline"] >= 0.20:
        reasons.append("CLICK_DECLINE")

    if row["session_decline"] >= 0.20:
        reasons.append("SESSION_DECLINE")

    if not reasons:
        reasons.append("NO_MAJOR_DECLINE")

    return "|".join(reasons)

baseline["reason_code"] = baseline.apply(get_reason_codes, axis=1)

# Output columns
output_cols = [
    "content_id",
    "baseline_score",
    "rank",
    "action",
    "reason_code",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"
]

baseline = baseline.sort_values("rank")

# Save CSV
import os
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline[output_cols].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows in queue:", len(baseline))
print("Missing scores:", baseline["baseline_score"].isna().sum())
print("Infinite scores:", np.isinf(baseline["baseline_score"]).sum())

print("\nTop 20:")
display(baseline[output_cols].head(20))

Saved: work/outputs/baseline_action_score.csv
Rows in queue: 30000
Missing scores: 0
Infinite scores: 0

Top 20:


,content_id,baseline_score,rank,action,reason_code,impressions_last_30d,impressions_prev_30d,clicks_last_30d,clicks_prev_30d,sessions_last_30d,sessions_prev_30d
1961,content_e2737c796b98,1.0,1,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,5,0,1,0,1
7618,content_3d28f4002b44,1.0,2,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,46,0,1,0,1
7774,content_22140cc4c8c4,1.0,3,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,5,0,1,0,1
9971,content_87a771e1dafb,1.0,4,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,20,0,1,0,1
11725,content_4c570d0171fa,1.0,5,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,6,0,1,0,2
12104,content_727fd10b04fc,1.0,6,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,14,0,1,0,3
12373,content_5af804ae0cd7,1.0,7,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,2,0,1,0,1
12534,content_81353b21b26a,1.0,8,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,2,0,1,0,6
16013,content_d7ac1351376a,1.0,9,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,76,0,1,0,3
17870,content_080489312982,1.0,10,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,0,73,0,1,0,3


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
I will review the top 20 items as a human-review queue. For each item, the action and reason code explain why it was prioritized. The confidence note will remind the reviewer that the baseline is directional, and the item should be checked against the underlying content before any refresh decision. A pick can be wrong when the observed decline is caused by seasonality, tracking changes, very low volume, or another factor not represented in the baseline.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline.head(20).copy()

top20["confidence_note"] = np.where(
    top20["baseline_score"] >= 0.60,
    "Higher baseline priority; verify before action.",
    "Directional priority; verify before action."
)

top20["what_could_make_it_wrong"] = (
    "Seasonality, low volume, tracking changes, or missing context"
)

review_cols = [
    "rank",
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "confidence_note",
    "what_could_make_it_wrong"
]

display(top20[review_cols])

,rank,content_id,baseline_score,action,reason_code,confidence_note,what_could_make_it_wrong
1961,1,content_e2737c796b98,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
7618,2,content_3d28f4002b44,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
7774,3,content_22140cc4c8c4,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
9971,4,content_87a771e1dafb,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
11725,5,content_4c570d0171fa,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
12104,6,content_727fd10b04fc,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
12373,7,content_5af804ae0cd7,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
12534,8,content_81353b21b26a,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
16013,9,content_d7ac1351376a,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."
17870,10,content_080489312982,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,Higher baseline priority; verify before action.,"Seasonality, low volume, tracking changes, or ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak picks are items that receive a high baseline score even though the underlying volumes are small or the decline is driven by only one signal. These picks should be treated cautiously because percentage changes can be unstable when the previous-period value is near zero. The baseline does not use trend_direction or trend_pct, and it does not use future windows, so these label-related fields are not leaked into the score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak-pick review
weak_picks = baseline[
    (baseline["baseline_score"] >= 0.60) &
    (
        (baseline["impressions_prev_30d"] < 10) |
        (baseline["clicks_prev_30d"] < 5) |
        (baseline["sessions_prev_30d"] < 5)
    )
].head(20)

print("Potential weak picks:", len(weak_picks))

display(
    weak_picks[
        [
            "content_id",
            "baseline_score",
            "action",
            "reason_code",
            "impressions_prev_30d",
            "clicks_prev_30d",
            "sessions_prev_30d"
        ]
    ]
)

# Leakage check
leakage_fields = [
    "trend_direction",
    "trend_pct"
]

used_features = [
    "impressions_change",
    "clicks_change",
    "sessions_change"
]

print("\nLeakage check:")
print("Label fields used in score:",
      [c for c in leakage_fields if c in used_features])

print("Future-window fields used in score: none")
print("Baseline uses only current vs previous 30-day signals.")

Potential weak picks: 20


,content_id,baseline_score,action,reason_code,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d
1961,content_e2737c796b98,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,5,1,1
7618,content_3d28f4002b44,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,46,1,1
7774,content_22140cc4c8c4,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,5,1,1
9971,content_87a771e1dafb,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,20,1,1
11725,content_4c570d0171fa,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,6,1,2
12104,content_727fd10b04fc,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,14,1,3
12373,content_5af804ae0cd7,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,2,1,1
12534,content_81353b21b26a,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,2,1,6
16013,content_d7ac1351376a,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,76,1,3
17870,content_080489312982,1.0,Review / Refresh,IMPRESSION_DECLINE|CLICK_DECLINE|SESSION_DECLINE,73,1,3



Leakage check:
Label fields used in score: []
Future-window fields used in score: none
Baseline uses only current vs previous 30-day signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.